# Semi-Supervised Training (SimCLR + Supervised) 
Unlike the other implementation where we do a mixed training of the unsupervised and the supervised training examples. Which did not seem to get much help from the unsupervised training - supervised only training seems to have performed better in some cases than the mixed training.

Now, we are instead going to try an approach where we do one approach after the other. That is, first we do unsupervised training using the unlabeled examples, and then we do supervised training on the labeled examples.

First we import everything necessary and load the model.

In [1]:
import open_clip
import torch
import json
import os
import gc
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import random
import matplotlib.pyplot as plt
from utils import plot_training_results
import torch.nn.functional as F
from tqdm import tqdm
import itertools
from utils import load_dataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR



device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# 1. Load the model, training transforms, and tokenizer
model_name = 'ViT-B-32'
pretrained_weights = 'openai'

model, base_train_transform, val_transform = open_clip.create_model_and_transforms(
    model_name, 
    pretrained=pretrained_weights
)

model = model.to(device)

tokenizer = open_clip.get_tokenizer(model_name)
print("Model and Tokenizer loaded successfully!")


/nobackup/marfr380/clip_venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


/nobackup/marfr380/clip_venv/lib/python3.10/site-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


Model and Tokenizer loaded successfully!


# Datasets
Now we can load our datasets and create our training and test splits.

In [2]:
splitA, splitB, splitC = load_dataset("data/dataset_rsicd.json", classes=True)

# Define split sizes
training_size = 0.9

# Reproducibility
random.seed(42)

# Shuffle our data
random.shuffle(splitB)
splitIdxA = int(len(splitA) * training_size) 
splitIdxB = int(len(splitB) * training_size) 

splitA_tr = splitA[:splitIdxA]
splitA_te = splitA[splitIdxA:]

splitB_tr = splitB[:splitIdxB]
splitB_te = splitB[splitIdxB:]

# Create Dataloaders and Transformations

In [ ]:
import time
import torch
import torch.nn.functional as F
from tqdm import tqdm

def info_nce_loss(features_1, features_2, logits_scale):
    # L2 Normalization (L_p, but p is 2 by default)
    feat_1 = F.normalize(features_1)
    feat_2 = F.normalize(features_2)

    # Calculate similarity scores
    # [N, Dim] @ [Dim, N] -> [N, N] similarity matrix
    logits = logits_scale * feat_1 @ feat_2.T

    # Define ground truth labels
    # The diagonal is our target, so for every row, the target is at the same index as the row.
    labels = torch.arange(len(logits)).to(device)
    
    # Calculate bidirectional cross-entropy loss.
    loss_1 = F.cross_entropy(logits, labels)
    loss_2 = F.cross_entropy(logits.T, labels)

    # Return the average of the losses
    return (loss_1 + loss_2) / 2.0

def train_unsupervised(model, unlabeled_loader, optimizer, scheduler, epochs, clip_norm=1.0):
    # Freeze text encoder to save memory/compute
    for param in model.transformer.parameters():
        param.requires_grad = False
        
    for epoch in range(epochs):
        model.train()
        pbar = tqdm(unlabeled_loader, desc=f"Unsupervised Epoch {epoch+1}/{epochs}")
        
        for view_1, view_2 in pbar:
            view_1, view_2 = view_1.to(device), view_2.to(device)
            
            optimizer.zero_grad()
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                # Only use the image encoder
                feat_1 = model.encode_image(view_1)
                feat_2 = model.encode_image(view_2)
                
                loss = info_nce_loss(feat_1, feat_2, model.logit_scale.exp())
                
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip_norm)
            optimizer.step()
            
            # GPU Usage calculation
            if torch.cuda.is_available():
                used_mem = torch.cuda.max_memory_allocated(device) / 1024**3
                total_mem = torch.cuda.get_device_properties(device).total_memory / 1024**3
                mem_report = f"{used_mem:.1f}/{total_mem:.1f}GB"
            else:
                mem_report = "N/A"

            pbar.set_postfix({
                'Unsup Loss': f"{loss.item():.3f}",
                'GPU': mem_report
            })
            
        scheduler.step()
    
    # Save the model after pre-training!
    torch.save(model.state_dict(), "clip_vision_pretrained.pt")
    return model


@torch.no_grad()
def evaluate_clip_loss(model, val_loader, logits_scale):
    model.eval()
    val_loss = 0.0
    num_batches = 0

    for images, tokens in val_loader:
        images, tokens = images.to(device), tokens.to(device)
        
        feat_img = model.encode_image(images)
        feat_tok = model.encode_text(tokens)

        loss = info_nce_loss(feat_img, feat_tok, logits_scale)

        val_loss += loss.item()
        num_batches += 1
    
    return val_loss / num_batches


@torch.no_grad()
def evaluate_zero_shot(model, dataloader, class_names, tokenizer):
    model.eval()

    prompts = [f"A satellite image of a {cls_name.lower()}." for cls_name in class_names]
    text_tokens = tokenizer(prompts).to(device)

    text_features = model.encode_text(text_tokens)
    text_features = F.normalize(text_features, dim=-1)

    correct = 0
    total = 0

    pbar = tqdm(dataloader, desc="Zero-Shot Evaluation")
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        
        # Encode and normalize images
        image_features = model.encode_image(images)
        image_features = F.normalize(image_features, dim=-1)
        
        # Calculate Cosine Similarity
        # Shape: [Batch_Size, Num_Classes]
        similarity = image_features @ text_features.T 
        
        # The predicted class is the one with the highest similarity score
        predictions = similarity.argmax(dim=-1)
        
        # Count correct predictions
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    final_accuracy = correct / total
    return final_accuracy


def train_supervised(model, labeled_loader, val_loader, optimizer, scheduler, epochs=10, freeze_text=False):
    
    # --- Handle Text Encoder Freezing ---
    if freeze_text:
        print("Freezing text encoder...")
        for param in model.transformer.parameters():
            param.requires_grad = False
    else:
        print("Unfreezing text encoder...")
        for param in model.transformer.parameters():
            param.requires_grad = True

    for epoch in range(epochs):
        model.train()
        
        # If text is frozen, ensure it stays in eval mode (disables dropout, etc.)
        if freeze_text:
            model.transformer.eval()
            
        pbar = tqdm(labeled_loader, desc=f"Supervised Epoch {epoch+1}/{epochs}")
        epoch_loss = 0
        
        for img, txt in pbar:
            img, txt = img.to(device), txt.to(device)
            
            optimizer.zero_grad()
            
            # Using mixed precision
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                feat_img = model.encode_image(img)
                feat_txt = model.encode_text(txt)
                
                # model.logit_scale is a learned parameter in CLIP
                loss = info_nce_loss(feat_img, feat_txt, model.logit_scale.exp())
                
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            
            # GPU Usage calculation
            if torch.cuda.is_available():
                used_mem = torch.cuda.max_memory_allocated(device) / 1024**3
                total_mem = torch.cuda.get_device_properties(device).total_memory / 1024**3
                mem_report = f"{used_mem:.1f}/{total_mem:.1f}GB"
            else:
                mem_report = "N/A"
            
            current_lr = scheduler.get_last_lr()[0]
            pbar.set_postfix({
                'Sup Loss': f"{loss.item():.3f}", 
                'LR': f"{current_lr:.2e}",
                'GPU': mem_report
            })
            
        scheduler.step()
            
        # Evaluate
        val_loss = evaluate_clip_loss(model, val_loader, model.logit_scale.exp())
        print(f"Epoch {epoch+1} | Train Loss: {epoch_loss/len(labeled_loader):.4f} | Val Loss: {val_loss:.4f}")

    return model

In [4]:
from rsiddataset import RSICDUnlabeledDataset, RSICDLabeledDataset, RSICDClassificationDataset

image_dir = "data/RSICD_images"


# Extract the exact normalization layer from the official CLIP transform
clip_normalize = base_train_transform.transforms[-1] 

simclr_transform = T.Compose([
    # 1. Less aggressive crop. Ensure at least 40% of the image remains 
    # so the overall "scene" context (like 'airport' or 'stadium') is preserved.
    T.RandomResizedCrop(224, scale=(0.2, 0.7)), 
    
    # 2. Geometric transformations are practically infinite free data
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.5),
    # Using reflect padding avoids black borders when rotating!
    T.RandomRotation(degrees=180, interpolation=T.InterpolationMode.BILINEAR, expand=False),
    
    # 3. Color: Alter Brightness/Contrast (simulates clouds/haze/sun angle)
    # BUT keep Saturation/Hue changes very low.
    T.RandomApply([T.ColorJitter(brightness=0.8, contrast=0.8, saturation=0.2, hue=0.0)], p=0.8),
    
    T.RandomApply([T.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0))], p=0.5),
    
    T.ToTensor(),
    
    # 5. Random Erasing helps prevent the model from memorizing single pixels
    T.RandomErasing(p=0.2, scale=(0.02, 0.1)),
    
    clip_normalize
])

training_a = RSICDUnlabeledDataset(splitA_tr, img_dir=image_dir, transform=simclr_transform)
test_a = RSICDUnlabeledDataset(splitA_te, img_dir=image_dir, transform=simclr_transform)

training_b = RSICDLabeledDataset(splitB_tr, img_dir=image_dir, transform=simclr_transform, tokenizer=tokenizer)
test_b     = RSICDLabeledDataset(splitB_te, img_dir=image_dir, transform=base_train_transform, tokenizer=tokenizer)

unique_classes = sorted(list(set(item['class'] for item in splitC)))
zs_dataset = RSICDClassificationDataset(splitC, img_dir=image_dir, transform=val_transform, class_names=unique_classes)

loader_args = {
    "batch_size": 256+128,
    "num_workers": 8,
    "pin_memory": True,
    "drop_last": True,
}

loader_A = DataLoader(training_a, shuffle=True, **loader_args)

loader_args['batch_size'] = 256
loader_B = DataLoader(training_b, shuffle=True, **loader_args)
val_loader_B = DataLoader(test_b, batch_size=256, shuffle=False, drop_last=False)

zs_loader = DataLoader(zs_dataset, batch_size=64, shuffle=False, num_workers=4)

# Training
Now we run the unsupervised training on only the image encoder, and then train the full model on the labeled data.

In [5]:
UNSUPERVISED_EPOCHS = 5
SUPERVISED_EPOCHS = 3

accuracy_0 = evaluate_zero_shot(model, zs_loader, unique_classes, tokenizer)
print(f"\n✅ Accuracy before training (Base model): {accuracy_0 * 100:.2f}%\n")

# ==========================================
# PHASE 1: UNSUPERVISED PRE-TRAINING
# ==========================================


print("\n--- Starting Phase 1: Unsupervised Pre-training ---")

vision_params = [p for p in model.visual.parameters() if p.requires_grad]
vision_params.append(model.logit_scale)

optimizer_1 = AdamW(vision_params, lr=5e-7, weight_decay=0.05)
scheduler_1 = CosineAnnealingLR(optimizer_1, T_max=UNSUPERVISED_EPOCHS, eta_min=1e-8)

model = train_unsupervised(
    model=model, 
    unlabeled_loader=loader_A, # matches function signature
    optimizer=optimizer_1, 
    scheduler=scheduler_1, 
    epochs=UNSUPERVISED_EPOCHS
)

# Evaluate and print after Unsupervised
accuracy_1 = evaluate_zero_shot(model, zs_loader, unique_classes, tokenizer)
print(f"\n✅ Accuracy after Phase 1 (Unsupervised): {accuracy_1 * 100:.2f}%\n")

# ==========================================
# 🧹 MEMORY CLEANUP BEFORE PHASE 2
# ==========================================
print("🧹 Clearing GPU Memory...")
del optimizer_1
del scheduler_1
del vision_params
gc.collect()
torch.cuda.empty_cache()

if torch.cuda.is_available():
    used_mem = torch.cuda.memory_allocated(device) / 1024**3
    print(f"Current GPU Memory Usage after cleanup: {used_mem:.2f} GB")

# ==========================================
# PHASE 2: SUPERVISED FINE-TUNING
# ==========================================
print("--- Starting Phase 2: Supervised Fine-tuning ---")

# Ensure text parameters are unfrozen BEFORE giving them to the optimizer
for param in model.transformer.parameters():
    param.requires_grad = True

all_params = [p for p in model.parameters() if p.requires_grad]
optimizer_2 = AdamW(all_params, lr=1e-5, weight_decay=0.01) # Notice lower LR!
scheduler_2 = CosineAnnealingLR(optimizer_2, T_max=SUPERVISED_EPOCHS, eta_min=1e-7)

model = train_supervised(
    model=model, 
    labeled_loader=loader_B,  # matches function signature
    val_loader=val_loader_B, 
    num_classes=unique_classes, # passed for your inner zero-shot evaluations
    optimizer=optimizer_2, 
    scheduler=scheduler_2, 
    epochs=SUPERVISED_EPOCHS
)

# Evaluate and print after Supervised
accuracy_2 = evaluate_zero_shot(model, zs_loader, unique_classes, tokenizer)
print(f"\n✅ Accuracy after Phase 2 (Supervised): {accuracy_2 * 100:.2f}%")

# ==========================================
# FINAL SUMMARY
# ==========================================
print("\n" + "="*40)
print("🎯 TRAINING SUMMARY")
print("="*40)
print(f"Base Model            : {accuracy_0 * 100:.2f}%")
print(f"After Unsupervised    : {accuracy_1 * 100:.2f}%")
print(f"After Supervised      : {accuracy_2 * 100:.2f}%")
print("="*40)

Zero-Shot Evaluation: 100%|██████████| 18/18 [00:01<00:00, 13.82it/s]



✅ Accuracy before training (Base model): 50.32%


--- Starting Phase 1: Unsupervised Pre-training ---


Zero-Shot Evaluation: 100%|██████████| 18/18 [00:01<00:00, 13.98it/s]



✅ Accuracy after Phase 1 (Unsupervised): 51.24%

🧹 Clearing GPU Memory...
Current GPU Memory Usage after cleanup: 0.91 GB
--- Starting Phase 2: Supervised Fine-tuning ---


Supervised Epoch 1/3: 100%|██████████| 19/19 [00:11<00:00,  1.62it/s, Sup Loss=2.861, LR=1.00e-05, GPU=14.3/19.5GB]


...

Epoch 2 | Train Loss: 2.5695 | Val Loss: 2.6916


Supervised Epoch 3/3: 100%|██████████| 19/19 [00:10<00:00,  1.75it/s, Sup Loss=2.236, LR=2.58e-06, GPU=14.3/19.5GB]


Epoch 3 | Train Loss: 2.3366 | Val Loss: 2.6828


Zero-Shot Evaluation: 100%|██████████| 18/18 [00:01<00:00, 14.53it/s]


✅ Accuracy after Phase 2 (Supervised): 80.70%

🎯 TRAINING SUMMARY
Base Model            : 50.32%
After Unsupervised    : 51.24%
After Supervised      : 80.70%


In [ ]:
model_save_dir = "/nobackup/marfr380/models"
save_path = os.path.join(model_save_dir, "clip_best_model_3.pt")
torch.save(model.state_dict(), save_path)
print(f"🌟 Model successfully saved to: {save_path}")

🌟 Model successfully saved to: /nobackup/marfr380/models/clip_best_model_2.pt
